# Chunking Strategy Evaluation

Compares six chunking strategies against MiniLM-L6-v2 embeddings using hand-crafted ground truth.

## Strategies
| Strategy | Input | Notes |
|---|---|---|
| `char_limit` | flat text list | sequential baseline |
| `element_type` | Unstructured elements | header-bounded, context prefix |
| `sentence_sliding` | flat text list | window=5, overlap=2 |
| `paragraph` | flat text list | split on `\n\n`, prose only |
| `token_aware` | flat text list | ≤200 MiniLM tokens |
| `table_aware` | page dicts + Unstructured elements | routing layer over `char_limit` |

## Pipeline
```
PDFs
 ├─ Numeric density → page classification (table / prose)
 ├─ Table pages  → Docling JSON (pre-baked, docling_test env)
 └─ Prose pages  → Unstructured fast strategy (rag env)
       ↓
Chunking strategies (this notebook, rag env)
       ↓
MiniLM embeddings → Recall@k, MRR vs ground_truth.json
```

## Prerequisites
- `ground_truth.json` in the repo root (see format below)
- Docling JSON files already extracted (see Cell 3 for the extraction script to run in `docling_test` env)
- `rag` conda env active with: `sentence-transformers`, `unstructured`, `nltk`, `transformers`, `pandas`, `numpy`

## When transferring to pipeline, make sure to avoid the necessity of a `gt_source_to_label`

## 0. Imports and config

In [ ]:
import torch

In [2]:
import os
from pathlib import Path

# Set CWD to repo root so utils.py and data/ paths resolve correctly.
# __vsc_ipynb_file__ is injected by VS Code when running a notebook.
_nb_file = globals().get("__vsc_ipynb_file__")
if _nb_file:
    _repo_root = Path(_nb_file).resolve().parents[2]
    os.chdir(_repo_root)
print(f"CWD: {Path.cwd()}")

CWD: C:\Users\elimo\problem-set-2-elimossmarks11


In [6]:
import json
import re
import sys

import nltk
import numpy as np
import pandas as pd
import pickle 
from nltk.tokenize import sent_tokenize
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer
from unstructured.partition.pdf import partition_pdf
import pdfplumber

# CWD is already the repo root (set by the cell above)
ROOT = Path(".").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from utils import (
    NUMERIC_DENSITY_THRESHOLD,
    bootstrap_runtime_env,
    chunk_by_char_limit,
    chunk_by_element_type,
    chunk_sentences_sliding,
    count_tokens,
    numeric_density,
)

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
bootstrap_runtime_env()

print("Imports OK")

Imports OK


In [ ]:
# ── paths ────────────────────────────────────────────────────────────────────
PDF_DIR            = ROOT / "data" / "chunking_test_pdfs"
DOCLING_JSON_DIR   = ROOT / "data" / "interim" / "docling_tests"
GROUND_TRUTH_PATH  = ROOT / "ground_truth.json"
RESULTS_DIR        = ROOT / "data" / "interim" / "chunking_eval"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ── document labels → PDF filenames ──────────────────────────────────────────
DOCUMENTS = {
    "antam":         "ANTAM AR2016.pdf",
    "angloamerican": "AR2019.pdf",
}

# ── evaluation scope ─────────────────────────────────────────────────────────
# Set to a DOCUMENTS key to evaluate one company only, or None to evaluate all.
EVAL_COMPANY: str | None = "angloamerican"

# Filtered view of DOCUMENTS — all processing loops use this
EVAL_DOCUMENTS = (
    {EVAL_COMPANY: DOCUMENTS[EVAL_COMPANY]} if EVAL_COMPANY else dict(DOCUMENTS)
)

# ── chunking hyperparameters ─────────────────────────────────────────────────
CHAR_LIMIT      = 1000   # used by char_limit, element_type, table_aware base
TOKEN_LIMIT     = 200    # conservative MiniLM limit (hard cap: 256)
WINDOW_SIZE     = 5      # sentence sliding
OVERLAP         = 2      # sentence sliding
EVAL_K          = 5      # Recall@k

# ── numeric density threshold imported from utils.py ─────────────────────────
print(f"NUMERIC_DENSITY_THRESHOLD = {NUMERIC_DENSITY_THRESHOLD}")

print("Config OK")
print(f"  PDFs:         {PDF_DIR}")
print(f"  Docling JSON: {DOCLING_JSON_DIR}")
print(f"  Ground truth: {GROUND_TRUTH_PATH}")
print(f"  Results:      {RESULTS_DIR}")
print(f"  Eval company: {EVAL_COMPANY or 'all'}")
print(f"  Documents:    {list(EVAL_DOCUMENTS.keys())}")

## 1. Classify pages by numeric density

Use pdfplumber (cheap) to compute numeric density per page and identify table pages.
Writes `{label}_table_pages.json` to `DOCLING_JSON_DIR` — this is the handoff artifact
that `extract_docling.py` reads to know which pages to convert.

In [ ]:
table_page_indices = {}  # label → list[int] (0-indexed page indices)

for label, filename in EVAL_DOCUMENTS.items():
    pdf_path = PDF_DIR / filename
    if not pdf_path.exists():
        print(f"SKIP {label}: {pdf_path} not found")
        continue

    print(f"\nClassifying {label} ...")
    with pdfplumber.open(pdf_path) as pdf:
        indices = []
        for i, page in enumerate(pdf.pages):
            words = page.extract_words()
            density = numeric_density(words)
            if density >= NUMERIC_DENSITY_THRESHOLD:
                indices.append(i)
        table_page_indices[label] = indices

    # Write handoff artifact for extract_docling.py
    out_path = DOCLING_JSON_DIR / f"{label}_table_pages.json"
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(json.dumps(indices), encoding="utf-8")

    print(f"  {len(indices)} table pages out of {len(pdf.pages)} total")
    print(f"  Written to {out_path}")

# Pickle checkpoint
_pkl_path = RESULTS_DIR / "table_page_indices.pkl"
with open(_pkl_path, "wb") as f:
    pickle.dump(table_page_indices, f)
print(f"\n✓ Pickled table_page_indices → {_pkl_path}")

print("\nClassification complete.")

## 2. Docling extraction (run once in `docling_test` env)

**Do not run this cell in your `rag` env** — Docling conflicts with Unstructured on `transformers`.

After running step 1 above, switch to `docling_test` env and run:
```
conda activate docling_test && python testing/chunking_tests/extract_docling.py
```
This reads `{label}_table_pages.json` (from step 1) and converts **only** the identified
table pages with Docling, writing `{label}_docling.json`.

In [ ]:
# Verify Docling JSON files exist before proceeding
print("Checking Docling JSON files in:", DOCLING_JSON_DIR)
for label in EVAL_DOCUMENTS:
    path = DOCLING_JSON_DIR / f"{label}_docling.json"
    print(f"  {label}: {'✓ found' if path.exists() else '✗ NOT FOUND — run extract_docling.py'}")


def load_docling_pages(label: str) -> list[dict]:
    """Load pre-baked Docling JSON for one document."""
    path = DOCLING_JSON_DIR / f"{label}_docling.json"
    if not path.exists():
        raise FileNotFoundError(
            f"Docling JSON not found: {path}\n"
            f"Run testing/chunking_tests/extract_docling.py in your docling_test env first."
        )
    pages = json.loads(path.read_text(encoding="utf-8"))
    print(f"  [{label}] Loaded {len(pages)} Docling table pages")
    return pages


def load_unstructured_elements(label: str, pdf_path: Path) -> list:
    """Partition prose pages with Unstructured fast strategy."""
    print(f"  [{label}] Partitioning with Unstructured (fast) ...")
    elements = partition_pdf(
        filename=str(pdf_path),
        strategy="fast",
    )
    print(f"  [{label}] {len(elements)} elements returned")
    return elements


# Load both sources for each document
docling_pages   = {}   # label → list[page_dict]
unstruct_elems  = {}   # label → list[Element]

for label, filename in EVAL_DOCUMENTS.items():
    pdf_path = PDF_DIR / filename
    print(f"\nLoading {label} ...")
    docling_pages[label]  = load_docling_pages(label)
    unstruct_elems[label] = load_unstructured_elements(label, pdf_path)

# Pickle checkpoint — Unstructured partitioning is the slowest step
for label in EVAL_DOCUMENTS:
    docling_pkl_path  = RESULTS_DIR / f"{label}_docling_pages.pkl"
    unstruct_pkl_path = RESULTS_DIR / f"{label}_unstruct_elems.pkl"
    with open(docling_pkl_path, "wb") as f:
        pickle.dump(docling_pages[label], f)
    with open(unstruct_pkl_path, "wb") as f:
        pickle.dump(unstruct_elems[label], f)
    print(f"  ✓ Pickled {label} → {docling_pkl_path.name}, {unstruct_pkl_path.name}")

print("\nAll documents loaded and checkpointed.")

## 3. New chunking strategies

Three strategies not yet in `utils.py`: `paragraph`, `token_aware`, and `table_aware`.

`table_aware` is a **routing layer**: table pages become atomic single chunks; prose pages are delegated to `char_limit` as the base strategy.

In [52]:
tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
print(f"Tokenizer loaded: {tokenizer.__class__.__name__}")
print(f"Token limit in use: {TOKEN_LIMIT} (model hard cap: 512, practical cap: 256)")

Tokenizer loaded: BertTokenizerFast
Token limit in use: 200 (model hard cap: 512, practical cap: 256)


In [ ]:
def chunk_by_paragraph(
    texts: list[str],
    min_chars: int = 50,
    source_label: str = "",
) -> list[dict]:
    """Split on blank lines (\\n\\n); discard fragments shorter than min_chars.

    Designed for pdfplumber / Unstructured plain-text output where paragraph
    boundaries are preserved as double newlines. Does NOT apply to Docling
    table-page markdown — use table_aware for those pages.
    """
    chunks = []
    for text in texts:
        paragraphs = [p.strip() for p in re.split(r"\n{2,}", text)]
        for para in paragraphs:
            if len(para) < min_chars:
                continue
            chunks.append({
                "id":       f"para_{len(chunks):04d}",
                "text":     para,
                "strategy": "paragraph",
                "source":   source_label,
            })
    return chunks


def chunk_by_token_limit(
    texts: list[str],
    tokenizer,
    token_limit: int = 200,
    source_label: str = "",
) -> list[dict]:
    """Chunk by sentence accumulation, flushing when the next sentence would
    exceed token_limit.

    Uses the MiniLM tokenizer directly so chunk boundaries respect the model's
    actual subword vocabulary rather than character-count proxies.
    Single sentences that exceed token_limit are emitted as-is with a warning
    rather than being silently truncated.
    """
    all_sentences = []
    for text in texts:
        all_sentences.extend(sent_tokenize(text))

    chunks = []
    buffer: list[str] = []
    buffer_tokens = 0

    for sent in all_sentences:
        sent_tokens = len(tokenizer.encode(sent, add_special_tokens=False))

        if sent_tokens > token_limit:
            # Flush existing buffer first
            if buffer:
                chunks.append({
                    "id":          f"tok_{len(chunks):04d}",
                    "text":        " ".join(buffer),
                    "strategy":    "token_aware",
                    "source":      source_label,
                    "token_count": buffer_tokens,
                })
                buffer, buffer_tokens = [], 0
            # Emit oversized sentence alone with warning
            print(f"  [token_aware] WARNING: single sentence exceeds limit "
                  f"({sent_tokens} tokens). Emitting as-is.")
            chunks.append({
                "id":          f"tok_{len(chunks):04d}",
                "text":        sent,
                "strategy":    "token_aware",
                "source":      source_label,
                "token_count": sent_tokens,
            })
            continue

        if buffer_tokens + sent_tokens > token_limit and buffer:
            chunks.append({
                "id":          f"tok_{len(chunks):04d}",
                "text":        " ".join(buffer),
                "strategy":    "token_aware",
                "source":      source_label,
                "token_count": buffer_tokens,
            })
            buffer, buffer_tokens = [], 0

        buffer.append(sent)
        buffer_tokens += sent_tokens

    if buffer:
        chunks.append({
            "id":          f"tok_{len(chunks):04d}",
            "text":        " ".join(buffer),
            "strategy":    "token_aware",
            "source":      source_label,
            "token_count": buffer_tokens,
        })

    return chunks


def _extract_table_context(text: str) -> str:
    """Extract ## headers from the top of Docling markdown to build a context prefix.

    Returns a string like:
        TABLE CONTEXT: FINANCIAL PERFORMANCE > 2. FINANCIAL PERFORMANCE BY SEGMENT
    or empty string if no headers found.
    """
    headers = []
    for line in text.split("\n"):
        stripped = line.strip()
        if stripped.startswith("## "):
            headers.append(stripped.lstrip("# ").strip())
        elif stripped.startswith("| ") or (stripped and not stripped.startswith("#")):
            # Stop once we hit table content or non-header prose
            break
    if not headers:
        return ""
    return "TABLE CONTEXT: " + " > ".join(headers)


def _parse_pipe_row(row: str) -> list[str]:
    """Split a markdown pipe row into cell values, stripping whitespace."""
    cells = row.strip().strip("|").split("|")
    return [c.strip() for c in cells]


def _linearize_row(header_row: str, data_row: str) -> str:
    """Convert a header + data row into natural language.

    Example:
        header: | US$ million | Group revenue | Underlying EBIT |
        data:   | De Beers    | 5,207         | 4,149           |
        →  "De Beers: Group revenue = 5,207, Underlying EBIT = 4,149"
    """
    headers = _parse_pipe_row(header_row)
    values  = _parse_pipe_row(data_row)

    # First column is typically the row label
    row_label = values[0] if values else ""
    pairs = []
    for col, val in zip(headers[1:], values[1:]):
        if col and val:
            pairs.append(f"{col} = {val}")

    if row_label and pairs:
        return f"{row_label}: {', '.join(pairs)}"
    elif row_label:
        return row_label
    elif pairs:
        return ", ".join(pairs)
    return ""


def _split_table_rows(text: str) -> list[dict]:
    """Parse Docling markdown into linearized per-row chunks.

    Each data row is converted to natural language:
        "De Beers: Group revenue = 5,207, Underlying EBIT = 4,149"
    prefixed with context headers from the page.

    Returns a list of dicts with keys:
        - "row_text": linearized chunk text (context prefix + NL row)
        - "is_row": True if this is a data row, False for non-table prose
    """
    lines = text.split("\n")
    context_prefix = _extract_table_context(text)

    header_row: str | None = None
    separator_row: str | None = None
    results: list[dict] = []
    prose_buffer: list[str] = []

    for line in lines:
        stripped = line.strip()

        # Skip markdown headers — already captured in context_prefix
        if stripped.startswith("#"):
            continue

        # Skip image placeholders
        if stripped.startswith("<!-- image"):
            continue

        # Table line detection
        if stripped.startswith("|"):
            # Separator row (|---|---|)
            if re.match(r"^\|[\s\-:|]+\|$", stripped):
                separator_row = stripped
                continue

            # If no header row yet, this is the header
            if header_row is None:
                header_row = stripped
                continue

            # Data row — linearize into natural language
            linearized = _linearize_row(header_row, stripped)
            if linearized:
                row_text = f"{context_prefix}\n{linearized}" if context_prefix else linearized
                results.append({
                    "row_text": row_text,
                    "is_row": True,
                })
        else:
            # Non-table content: could be a new table starting, or prose
            if stripped:
                # Reset header tracking — next table line starts a new table
                if header_row is not None and not stripped.startswith("|"):
                    header_row = None
                    separator_row = None
                prose_buffer.append(stripped)

    # Emit collected prose (if any) as a single chunk
    if prose_buffer:
        prose_text = " ".join(prose_buffer)
        if len(prose_text) >= 50:
            prefix = f"{context_prefix}\n\n{prose_text}" if context_prefix else prose_text
            results.append({
                "row_text": prefix,
                "is_row": False,
            })

    return results


def chunk_table_aware(
    docling_pages: list[dict],
    unstructured_elements: list,
    char_limit: int = 1000,
    source_label: str = "",
) -> list[dict]:
    """Route table pages to per-row chunks; delegate prose pages to char_limit.

    Table pages (is_table=True in Docling JSON) are split into one chunk per
    table row. Each row chunk is prefixed with the table's context headers and
    column header row, giving MiniLM the semantic bridge between query terms
    (e.g. "underlying EBIT") and the row label (e.g. "De Beers").

    Prose pages are extracted from Unstructured elements filtered to pages
    NOT flagged as table pages, then chunked with chunk_by_char_limit.
    """
    table_page_nums = {p["page_num"] for p in docling_pages if p["is_table"]}
    chunks = []

    # -- table pages: one chunk per row, with column headers prepended --
    for page in docling_pages:
        if not page["is_table"]:
            continue
        text = page["text"].strip()
        if not text:
            continue

        row_chunks = _split_table_rows(text)
        for rc in row_chunks:
            chunks.append({
                "id":       f"tbl_{len(chunks):04d}",
                "text":     rc["row_text"],
                "strategy": "table_aware",
                "source":   source_label,
                "page_num": page["page_num"],
                "is_table": True,
                "is_row":   rc["is_row"],
            })

    # -- prose pages: filter Unstructured elements to non-table pages only --
    prose_elements = [
        el for el in unstructured_elements
        if getattr(el.metadata, "page_number", None) not in table_page_nums
    ]
    prose_texts = [
        (getattr(el, "text", None) or "").strip()
        for el in prose_elements
    ]
    prose_texts = [t for t in prose_texts if t]

    prose_chunks = chunk_by_char_limit(
        prose_texts,
        char_limit=char_limit,
        source_label=source_label,
    )
    # Re-ID prose chunks so IDs are globally unique within this strategy
    for pc in prose_chunks:
        pc["id"]       = f"tbl_p{len(chunks):04d}"
        pc["strategy"] = "table_aware"
        pc["is_table"] = False
        chunks.append(pc)

    n_table = sum(1 for c in chunks if c.get("is_table"))
    n_rows  = sum(1 for c in chunks if c.get("is_row"))
    print(f"  [table_aware] {len(chunks)} chunks total "
          f"({n_table} table [{n_rows} data rows], {len(chunks)-n_table} prose)")
    return chunks


print("New chunking strategies defined.")

New chunking strategies defined.


## 4. Generate all chunks

In [54]:
all_chunks = {}   # strategy_name → list[chunk_dict] (across evaluated documents)

for label in EVAL_DOCUMENTS:
    pages    = docling_pages[label]
    elements = unstruct_elems[label]

    # Flat text lists derived from Unstructured elements (prose pages only,
    # for strategies that don't need element objects)
    table_page_nums = {p["page_num"] for p in pages if p["is_table"]}
    prose_texts = [
        (getattr(el, "text", None) or "").strip()
        for el in elements
        if getattr(el.metadata, "page_number", None) not in table_page_nums
        and (getattr(el, "text", None) or "").strip()
    ]

    print(f"\n── {label} ──────────────────────────────────")

    # 1. char_limit (baseline) ------------------------------------------------
    c = chunk_by_char_limit(prose_texts, char_limit=CHAR_LIMIT, source_label=label)
    all_chunks.setdefault("char_limit", []).extend(c)
    print(f"  char_limit:      {len(c):4d} chunks")

    # 2. element_type (header-bounded) ----------------------------------------
    prose_elements = [
        el for el in elements
        if getattr(el.metadata, "page_number", None) not in table_page_nums
    ]
    c = chunk_by_element_type(prose_elements, char_limit=CHAR_LIMIT, source_label=label)
    all_chunks.setdefault("element_type", []).extend(c)
    print(f"  element_type:    {len(c):4d} chunks")

    # 3. sentence_sliding -----------------------------------------------------
    c = chunk_sentences_sliding(
        prose_texts,
        window_size=WINDOW_SIZE,
        overlap=OVERLAP,
        source_label=label,
    )
    all_chunks.setdefault("sentence_sliding", []).extend(c)
    print(f"  sentence_sliding:{len(c):4d} chunks")

    # 4. paragraph ------------------------------------------------------------
    c = chunk_by_paragraph(prose_texts, source_label=label)
    all_chunks.setdefault("paragraph", []).extend(c)
    print(f"  paragraph:       {len(c):4d} chunks")

    # 5. token_aware ----------------------------------------------------------
    c = chunk_by_token_limit(
        prose_texts,
        tokenizer=tokenizer,
        token_limit=TOKEN_LIMIT,
        source_label=label,
    )
    all_chunks.setdefault("token_aware", []).extend(c)
    print(f"  token_aware:     {len(c):4d} chunks")

    # 6. table_aware ----------------------------------------------------------
    c = chunk_table_aware(
        docling_pages=pages,
        unstructured_elements=elements,
        char_limit=CHAR_LIMIT,
        source_label=label,
    )
    all_chunks.setdefault("table_aware", []).extend(c)

print("\n── totals ───────────────────────────────────")
for strat, chunks in all_chunks.items():
    print(f"  {strat:<18s} {len(chunks):5d} chunks total")

# Pickle checkpoint
_pkl_path = RESULTS_DIR / "all_chunks.pkl"
with open(_pkl_path, "wb") as f:
    pickle.dump(all_chunks, f)
print(f"\n✓ Pickled all_chunks → {_pkl_path}")


── angloamerican ──────────────────────────────────
  char_limit:       865 chunks
  element_type:     949 chunks
  sentence_sliding:2450 chunks
  paragraph:       2539 chunks
  token_aware:      754 chunks
  [table_aware] 2348 chunks total (1483 table [1448 data rows], 865 prose)

── totals ───────────────────────────────────
  char_limit           865 chunks total
  element_type         949 chunks total
  sentence_sliding    2450 chunks total
  paragraph           2539 chunks total
  token_aware          754 chunks total
  table_aware         2348 chunks total

✓ Pickled all_chunks → C:\Users\elimo\problem-set-2-elimossmarks11\data\interim\chunking_eval\all_chunks.pkl


## 5. Chunk size diagnostics

Sanity-check chunk distributions before running evaluation. Red flags:
- Very high mean/median → chunks likely too large, may dilute relevant content
- Very low median with high max → strategy producing many tiny fragments
- `token_aware` max should be ≤ 200 by construction

In [55]:
diag_rows = []
for strat, chunks in all_chunks.items():
    char_counts  = [len(c["text"]) for c in chunks]
    token_counts = count_tokens([c["text"] for c in chunks], tokenizer)
    diag_rows.append({
        "strategy":      strat,
        "n_chunks":      len(chunks),
        "chars_mean":    int(np.mean(char_counts)),
        "chars_median":  int(np.median(char_counts)),
        "chars_max":     int(np.max(char_counts)),
        "tokens_mean":   int(np.mean(token_counts)),
        "tokens_median": int(np.median(token_counts)),
        "tokens_max":    int(np.max(token_counts)),
    })

diag_df = pd.DataFrame(diag_rows).set_index("strategy")
print("Chunk size diagnostics:")
display(diag_df)

# Warn if token_aware exceeds limit
tok_max = diag_df.loc["token_aware", "tokens_max"]
if tok_max > TOKEN_LIMIT:
    print(f"\n⚠ token_aware max tokens ({tok_max}) exceeds TOKEN_LIMIT ({TOKEN_LIMIT}) "
          "— oversized sentences emitted as-is (see warnings above).")
else:
    print(f"\n✓ token_aware max tokens ({tok_max}) within limit ({TOKEN_LIMIT}).")

Token indices sequence length is longer than the specified maximum sequence length for this model (664 > 512). Running this sequence through the model will result in indexing errors


Chunk size diagnostics:


,n_chunks,chars_mean,chars_median,chars_max,tokens_mean,tokens_median,tokens_max
strategy,,,,,,,
char_limit,865,832,878,3894,159,162,664
element_type,949,810,845,4010,156,160,698
sentence_sliding,2450,490,483,2335,94,91,425
paragraph,2539,264,208,3894,50,39,664
token_aware,754,955,962,1270,183,187,200
table_aware,2348,741,802,3949,222,179,1300



✓ token_aware max tokens (200) within limit (200).


## 6. Ground truth: load and resolve snippet → chunk IDs

In [56]:
def normalise(text: str) -> str:
    """Lowercase + collapse whitespace + fix common PDF artefacts."""
    t = text.lower().strip()
    # Fix hyphen-space from PDF line breaks: "business-as- usual" → "business-as-usual"
    t = re.sub(r"-\s+", "-", t)
    # Remove soft hyphens
    t = t.replace("\u00ad", "")
    # Collapse whitespace
    t = re.sub(r"\s+", " ", t)
    return t


# Minimum fraction of snippet characters that must align to a chunk
SNIPPET_COVERAGE_THRESHOLD = 0.6

# Minimum word-set overlap to attempt expensive SequenceMatcher
_WORD_OVERLAP_PREFILTER = 0.3


def snippet_coverage(snippet: str, chunk: str) -> float:
    """Fraction of *snippet* characters covered by matching blocks in chunk."""
    from difflib import SequenceMatcher

    s = SequenceMatcher(None, snippet, chunk, autojunk=False)
    matched_chars = sum(block.size for block in s.get_matching_blocks())
    return matched_chars / len(snippet) if snippet else 0.0


def _word_set(text: str) -> set[str]:
    """Extract word tokens for fast pre-filtering."""
    return set(text.split())


def resolve_snippet_to_ids(
    snippet: str,
    source: str,
    chunks: list[dict],
    norm_cache: dict[str, str],
    coverage_threshold: float = SNIPPET_COVERAGE_THRESHOLD,
) -> list[str]:
    """Find chunk IDs whose normalised text contains the normalised snippet."""
    norm_snippet = normalise(snippet)
    source_chunks = [c for c in chunks if c.get("source") == source]

    # 1. Exact substring match (preferred, fast)
    exact = [c["id"] for c in source_chunks if norm_snippet in norm_cache[c["id"]]]
    if exact:
        return exact

    # 2. Coverage fallback with word-overlap pre-filter
    snippet_words = _word_set(norm_snippet)
    if not snippet_words:
        return []

    fuzzy = []
    for c in source_chunks:
        norm_c = norm_cache[c["id"]]
        chunk_words = _word_set(norm_c)
        overlap = len(snippet_words & chunk_words) / len(snippet_words)
        if overlap < _WORD_OVERLAP_PREFILTER:
            continue
        cov = snippet_coverage(norm_snippet, norm_c)
        if cov >= coverage_threshold:
            fuzzy.append((cov, c["id"]))

    if fuzzy:
        fuzzy.sort(reverse=True)
        best_cov = fuzzy[0][0]
        return [cid for cov, cid in fuzzy if cov >= best_cov - 0.05]

    return []


# Map ground truth source filenames → chunk labels
_fn_to_label = {fn: lbl for lbl, fn in DOCUMENTS.items()}


def gt_source_to_label(src: str) -> str | None:
    """Map a ground truth `source` filename to the chunk source label."""
    if src in _fn_to_label:
        return _fn_to_label[src]
    # Suffix match: GT "AR2016.pdf" matches DOCUMENTS "ANTAM AR2016.pdf"
    for fn, lbl in _fn_to_label.items():
        if fn.endswith(src):
            return lbl
    return None


if not GROUND_TRUTH_PATH.exists():
    raise FileNotFoundError(
        f"{GROUND_TRUTH_PATH} not found.\n"
        "Create it with the format documented in Cell 0 before running evaluation."
    )

raw_gt = json.loads(GROUND_TRUTH_PATH.read_text(encoding="utf-8"))
print(f"Loaded {len(raw_gt)} ground truth entries from {GROUND_TRUTH_PATH}")

# Filter to a single company if EVAL_COMPANY is set
if EVAL_COMPANY is not None:
    raw_gt = [item for item in raw_gt if gt_source_to_label(item["source"]) == EVAL_COMPANY]
    print(f"Filtered to '{EVAL_COMPANY}': {len(raw_gt)} entries remaining")

# Resolve snippets to chunk IDs for each strategy
resolved_gt   = {}
unmatched_log = []

for strat, chunks in all_chunks.items():
    # Pre-normalise all chunk texts once per strategy
    norm_cache = {c["id"]: normalise(c["text"]) for c in chunks}

    entries = []
    for item in raw_gt:
        query   = item["query"]
        snippet = item["snippet"]
        source  = gt_source_to_label(item["source"])
        if source is None:
            unmatched_log.append((strat, query, f"unknown GT source: {item['source']}"))
            continue
        ids = resolve_snippet_to_ids(snippet, source, chunks, norm_cache)

        if not ids:
            unmatched_log.append((strat, query, snippet[:80]))
            continue

        entries.append({"query": query, "relevant_ids": ids})
    resolved_gt[strat] = entries

if unmatched_log:
    seen = set()
    unique_unmatched = []
    for strat, query, snip in unmatched_log:
        key = (query, snip)
        if key not in seen:
            seen.add(key)
            unique_unmatched.append((strat, query, snip))

    print(f"\n⚠ {len(unique_unmatched)} unique unmatched snippet(s):")
    for strat, query, snip in unique_unmatched:
        print(f"  query='{query[:50]}...' | snippet='{snip}...'")
else:
    print("\n✓ All snippets matched in all strategies.")


Loaded 8 ground truth entries from C:\Users\elimo\problem-set-2-elimossmarks11\ground_truth.json
Filtered to 'angloamerican': 6 entries remaining

⚠ 1 unique unmatched snippet(s):
  query='What was the underlying EBIT for De Beers in 2019...' | snippet='De Beers | 4,605 | 558 | (390) | 168...'


In [57]:
# ── Diagnostic: why are GT snippets unmatched? ───────────────────────────────
for item in raw_gt:
    query   = item["query"][:60]
    snippet = item["snippet"]
    src     = item["source"]
    label   = gt_source_to_label(src)
    norm_sn = normalise(snippet)

    print(f"\n{'='*80}")
    print(f"Query:   {query}...")
    print(f"Source:  {src} → label={label}")
    print(f"Snippet: {norm_sn[:100]}...")

    if label is None:
        print("  ❌ Source mapping failed!")
        continue

    # Check using first strategy's chunks
    strat_name = list(all_chunks.keys())[0]
    source_chunks = [c for c in all_chunks[strat_name] if c.get("source") == label]
    print(f"  Chunks with source='{label}' in '{strat_name}': {len(source_chunks)}")

    # Exact substring match
    exact_matches = [c["id"] for c in source_chunks if norm_sn in normalise(c["text"])]
    if exact_matches:
        print(f"  ✅ Exact match in {len(exact_matches)} chunk(s): {exact_matches[:3]}")
        continue

    # Find best coverage match
    best_cov = 0.0
    best_chunk = None
    for c in source_chunks:
        cov = snippet_coverage(norm_sn, normalise(c["text"]))
        if cov > best_cov:
            best_cov = cov
            best_chunk = c

    if best_chunk:
        status = "✅" if best_cov >= SNIPPET_COVERAGE_THRESHOLD else "❌"
        print(f"  {status} No exact match. Best snippet coverage: {best_cov:.3f} "
              f"(threshold: {SNIPPET_COVERAGE_THRESHOLD})")
        print(f"     Best chunk ID: {best_chunk['id']}")
        print(f"     Full chunk text (first 300 chars):")
        print(f"       {normalise(best_chunk['text'])[:300]}")
    else:
        print(f"  ❌ No chunks found for source '{label}'")


Query:   What are Anglo American's energy use targets for 2020?...
Source:  AR2019.pdf → label=angloamerican
Snippet: we will reduce our energy use by 8% by the end of 2020, against the 2016 business-as-usual (bau) pro...
  Chunks with source='angloamerican' in 'char_limit': 865
  ✅ Exact match in 1 chunk(s): ['char_0164']

Query:   What are Anglo American's GHG emissions targets for 2020?...
Source:  AR2019.pdf → label=angloamerican
Snippet: we will reduce our greenhouse gas (ghg) emissions by 22% relative to the 2016 bau projection, by the...
  Chunks with source='angloamerican' in 'char_limit': 865
  ✅ No exact match. Best snippet coverage: 1.000 (threshold: 0.6)
     Best chunk ID: char_0164
     Full chunk text (first 300 chars):
       our pgm operations are on course to achieve zero waste to landfill by the end of 2020 we made good progress incorporating environmental management into our operating model, including the development of an integrated data dashboard that will be ful

## 7. Embed and evaluate

In [58]:
print("Loading MiniLM ...")
minilm = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
print("Model loaded.")


def evaluate_strategy(
    strategy_name: str,
    chunks: list[dict],
    chunk_embs: np.ndarray,
    gt_entries: list[dict],
    k: int = 5,
) -> pd.DataFrame:
    """Compute Recall@k and MRR for each query using pre-computed embeddings."""
    if not gt_entries:
        print(f"  [{strategy_name}] No valid ground truth entries — skipping.")
        return pd.DataFrame()

    chunk_ids = [c["id"] for c in chunks]

    rows = []
    for entry in gt_entries:
        query       = entry["query"]
        relevant    = set(entry["relevant_ids"])
        q_vec       = minilm.encode(query, normalize_embeddings=True)
        sims        = np.dot(chunk_embs, q_vec)          # cosine sim (normalised)
        top_k_idx   = np.argsort(sims)[::-1][:k]
        retrieved   = [chunk_ids[i] for i in top_k_idx]

        hits        = len(relevant & set(retrieved))
        recall_at_k = hits / len(relevant) if relevant else 0.0

        mrr = 0.0
        for rank, rid in enumerate(retrieved, 1):
            if rid in relevant:
                mrr = 1.0 / rank
                break

        rows.append({
            "strategy":       strategy_name,
            "query":          query[:60] + "..." if len(query) > 60 else query,
            f"recall@{k}":    round(recall_at_k, 4),
            "mrr":            round(mrr, 4),
            "n_relevant_ids": len(relevant),
        })

    return pd.DataFrame(rows)


print("Evaluation function defined.")

Loading MiniLM ...
Model loaded.
Evaluation function defined.


## 9. Hybrid Retrieval (Dense + BM25)

In [66]:
import math
from collections import Counter

# ── BM25 implementation (no external dependencies) ───────────────────────────

# Standard BM25 hyperparameters
BM25_K1 = 1.5   # term-frequency saturation
BM25_B  = 0.75  # length normalisation

# RRF constant (standard value from Cormack et al. 2009)
RRF_K = 60


def _tokenize_bm25(text: str) -> list[str]:
    """Simple whitespace + lowercased tokenizer for BM25."""
    return re.findall(r"[a-z0-9]+", text.lower())


class BM25:
    """Minimal BM25 scorer over a pre-tokenized corpus."""

    def __init__(self, corpus_texts: list[str]) -> None:
        self.n_docs = len(corpus_texts)
        self.doc_tokens: list[list[str]] = [_tokenize_bm25(t) for t in corpus_texts]
        self.doc_lens = [len(d) for d in self.doc_tokens]
        self.avgdl = sum(self.doc_lens) / self.n_docs if self.n_docs else 1.0
        # Document frequency: how many docs contain each term
        self.df: dict[str, int] = Counter()
        for tokens in self.doc_tokens:
            for t in set(tokens):
                self.df[t] += 1

    def score(self, query: str) -> np.ndarray:
        """Return BM25 scores for all documents given a query string."""
        q_tokens = _tokenize_bm25(query)
        scores = np.zeros(self.n_docs)
        for qt in q_tokens:
            df_t = self.df.get(qt, 0)
            idf = math.log((self.n_docs - df_t + 0.5) / (df_t + 0.5) + 1.0)
            for i, doc_tokens in enumerate(self.doc_tokens):
                tf = doc_tokens.count(qt)
                if tf == 0:
                    continue
                num = tf * (BM25_K1 + 1)
                den = tf + BM25_K1 * (1 - BM25_B + BM25_B * self.doc_lens[i] / self.avgdl)
                scores[i] += idf * num / den
        return scores


def evaluate_strategy_hybrid(
    strategy_name: str,
    chunks: list[dict],
    chunk_embs: np.ndarray,
    gt_entries: list[dict],
    k: int = 5,
) -> pd.DataFrame:
    """Evaluate with Reciprocal Rank Fusion: dense (MiniLM) + sparse (BM25)."""
    if not gt_entries:
        print(f"  [{strategy_name}_hybrid] No valid ground truth entries — skipping.")
        return pd.DataFrame()

    chunk_ids = [c["id"] for c in chunks]
    corpus_texts = [c["text"] for c in chunks]
    bm25 = BM25(corpus_texts)
    n = len(chunk_ids)

    rows = []
    for entry in gt_entries:
        query    = entry["query"]
        relevant = set(entry["relevant_ids"])

        # Dense ranking
        q_vec      = minilm.encode(query, normalize_embeddings=True)
        dense_sims = np.dot(chunk_embs, q_vec)
        dense_rank = np.argsort(dense_sims)[::-1]

        # BM25 ranking
        bm25_scores = bm25.score(query)
        bm25_rank   = np.argsort(bm25_scores)[::-1]

        # RRF fusion: score(d) = 1/(RRF_K + rank_dense) + 1/(RRF_K + rank_bm25)
        rrf_scores = np.zeros(n)
        for rank_pos, doc_idx in enumerate(dense_rank):
            rrf_scores[doc_idx] += 1.0 / (RRF_K + rank_pos + 1)
        for rank_pos, doc_idx in enumerate(bm25_rank):
            rrf_scores[doc_idx] += 1.0 / (RRF_K + rank_pos + 1)

        top_k_idx = np.argsort(rrf_scores)[::-1][:k]
        retrieved = [chunk_ids[i] for i in top_k_idx]

        hits        = len(relevant & set(retrieved))
        recall_at_k = hits / len(relevant) if relevant else 0.0

        mrr = 0.0
        for rank, rid in enumerate(retrieved, 1):
            if rid in relevant:
                mrr = 1.0 / rank
                break

        rows.append({
            "strategy":       f"{strategy_name}_hybrid",
            "query":          query[:60] + "..." if len(query) > 60 else query,
            f"recall@{k}":    round(recall_at_k, 4),
            "mrr":            round(mrr, 4),
            "n_relevant_ids": len(relevant),
        })

    return pd.DataFrame(rows)


print("BM25 + hybrid evaluation function defined.")

BM25 + hybrid evaluation function defined.


In [59]:
# Pre-compute embeddings for all strategies (expensive — cached to pickle)
chunk_embeddings = {}  # strategy → np.ndarray

for strat, chunks in all_chunks.items():
    texts = [c["text"] for c in chunks]
    print(f"  [{strat}] Embedding {len(texts)} chunks ...")
    chunk_embeddings[strat] = minilm.encode(
        texts, normalize_embeddings=True, show_progress_bar=False
    )

# Pickle checkpoint
_pkl_path = RESULTS_DIR / "chunk_embeddings.pkl"
with open(_pkl_path, "wb") as f:
    pickle.dump(chunk_embeddings, f)
print(f"\n✓ Pickled chunk_embeddings → {_pkl_path}")

  [char_limit] Embedding 865 chunks ...
  [element_type] Embedding 949 chunks ...
  [sentence_sliding] Embedding 2450 chunks ...
  [paragraph] Embedding 2539 chunks ...
  [token_aware] Embedding 754 chunks ...
  [table_aware] Embedding 2348 chunks ...

✓ Pickled chunk_embeddings → C:\Users\elimo\problem-set-2-elimossmarks11\data\interim\chunking_eval\chunk_embeddings.pkl


In [67]:
result_frames = []

for strat, chunks in all_chunks.items():
    gt_entries = resolved_gt.get(strat, [])
    embs = chunk_embeddings[strat]
    # Dense-only evaluation
    df = evaluate_strategy(strat, chunks, embs, gt_entries, k=EVAL_K)
    if not df.empty:
        result_frames.append(df)
    # Hybrid (dense + BM25) evaluation
    df_hybrid = evaluate_strategy_hybrid(strat, chunks, embs, gt_entries, k=EVAL_K)
    if not df_hybrid.empty:
        result_frames.append(df_hybrid)

results_df = pd.concat(result_frames, ignore_index=True)
print(f"\nEvaluation complete: {len(results_df)} rows")


Evaluation complete: 62 rows


## 8. Results

In [68]:
# Per-query breakdown
print("Per-query results:")
display(
    results_df
    .sort_values(["query", f"recall@{EVAL_K}"], ascending=[True, False])
    .reset_index(drop=True)
)

Per-query results:


,strategy,query,recall@5,mrr,n_relevant_ids
0,char_limit,Does Anglo American account for downstream emi...,1.0,0.2000,1
1,char_limit_hybrid,Does Anglo American account for downstream emi...,1.0,0.2500,1
2,element_type,Does Anglo American account for downstream emi...,1.0,0.2000,1
3,element_type_hybrid,Does Anglo American account for downstream emi...,1.0,0.5000,1
4,paragraph_hybrid,Does Anglo American account for downstream emi...,1.0,0.2500,1
...,...,...,...,...,...
57,token_aware,What were Anglo American's Scope 1 Emissions i...,1.0,1.0000,1
58,token_aware_hybrid,What were Anglo American's Scope 1 Emissions i...,1.0,1.0000,1
59,table_aware,What were Anglo American's Scope 1 Emissions i...,1.0,0.2500,1
60,table_aware_hybrid,What were Anglo American's Scope 1 Emissions i...,1.0,0.3333,1


In [69]:
# Aggregate summary: mean Recall@k and MRR per strategy
summary_df = (
    results_df
    .groupby("strategy")[[f"recall@{EVAL_K}", "mrr"]]
    .mean()
    .round(4)
    .sort_values(f"recall@{EVAL_K}", ascending=False)
)

print(f"\nAggregate summary (mean over {len(raw_gt)} queries):")
display(summary_df)

best = summary_df.index[0]
print(f"\nBest strategy by mean Recall@{EVAL_K}: {best}")


Aggregate summary (mean over 6 queries):


,recall@5,mrr
strategy,,
element_type_hybrid,1.0,0.5333
element_type,1.0,0.3567
paragraph_hybrid,1.0,0.5500
paragraph,0.8,0.4167
token_aware,0.8,0.5400
token_aware_hybrid,0.8,0.5067
sentence_sliding_hybrid,0.7,0.3867
char_limit_hybrid,0.6,0.3167
char_limit,0.6,0.2900



Best strategy by mean Recall@5: element_type_hybrid


In [63]:
# Save results to CSV
per_query_path = RESULTS_DIR / "chunking_eval_per_query.csv"
summary_path   = RESULTS_DIR / "chunking_eval_summary.csv"

results_df.to_csv(per_query_path, index=False)
summary_df.to_csv(summary_path)

print(f"Results saved:")
print(f"  Per-query: {per_query_path}")
print(f"  Summary:   {summary_path}")

Results saved:
  Per-query: C:\Users\elimo\problem-set-2-elimossmarks11\data\interim\chunking_eval\chunking_eval_per_query.csv
  Summary:   C:\Users\elimo\problem-set-2-elimossmarks11\data\interim\chunking_eval\chunking_eval_summary.csv


## 9. Qualitative inspection

For any query, print the top-k chunks retrieved by each strategy side by side.
Useful for diagnosing *why* a strategy underperforms — e.g. table rows being split mid-row, or headers being stripped.

In [71]:
def inspect_query(
    query: str,
    k: int = 3,
    strategies: list[str] | None = None,
    hybrid: bool = False,
) -> None:
    """Print top-k retrieved chunks for a query across all (or selected) strategies.

    Args:
        hybrid: If True, use RRF fusion of dense (MiniLM) + sparse (BM25) rankings.
    """
    strategies = strategies or list(all_chunks.keys())
    q_vec = minilm.encode(query, normalize_embeddings=True)

    for strat in strategies:
        chunks = all_chunks[strat]
        ids    = [c["id"] for c in chunks]
        embs   = chunk_embeddings[strat]  # use pre-computed embeddings
        n      = len(ids)

        if hybrid:
            # Dense ranking
            dense_sims = np.dot(embs, q_vec)
            dense_rank = np.argsort(dense_sims)[::-1]

            # BM25 ranking
            bm25 = BM25([c["text"] for c in chunks])
            bm25_scores = bm25.score(query)
            bm25_rank   = np.argsort(bm25_scores)[::-1]

            # RRF fusion
            rrf_scores = np.zeros(n)
            for rank_pos, doc_idx in enumerate(dense_rank):
                rrf_scores[doc_idx] += 1.0 / (RRF_K + rank_pos + 1)
            for rank_pos, doc_idx in enumerate(bm25_rank):
                rrf_scores[doc_idx] += 1.0 / (RRF_K + rank_pos + 1)

            top_k = np.argsort(rrf_scores)[::-1][:k]
            scores = rrf_scores
            score_label = "rrf"
        else:
            scores = np.dot(embs, q_vec)
            top_k  = np.argsort(scores)[::-1][:k]
            score_label = "sim"

        mode = "HYBRID" if hybrid else "DENSE"
        print(f"\n{'═'*70}")
        print(f"  Strategy: {strat} [{mode}]  |  Query: {query[:60]}")
        print(f"{'═'*70}")
        for rank, idx in enumerate(top_k, 1):
            print(f"  Rank {rank} | id={ids[idx]} | {score_label}={scores[idx]:.4f}")
            print(f"  {chunks[idx]['text'][:300]}")
            print()


# Dense-only (current behaviour)
inspect_query("What was the underlying EBIT for De Beers in 2019?", strategies=["table_aware"])

# Hybrid — does BM25 surface the table chunk?
print("\n" + "▓"*70)
print("  HYBRID MODE")
print("▓"*70)
inspect_query("What was the underlying EBIT for De Beers in 2019?", strategies=["table_aware"], hybrid=True)


══════════════════════════════════════════════════════════════════════
  Strategy: table_aware [DENSE]  |  Query: What was the underlying EBIT for De Beers in 2019?
══════════════════════════════════════════════════════════════════════
  Rank 1 | id=tbl_p1757 | sim=0.6271
  Other The $0.1 billion decrease in underlying EBITDA was driven by lower volumes in response to weaker demand at the Group’s associate, Cerrejón, as well as Voorspoed and Victor mines (De Beers) ceasing operations. Also included are charges to the income statement in respect of environmental restora

  Rank 2 | id=tbl_p2291 | sim=0.5700
  Mining EBITDA margin The Mining EBITDA margin is derived from the Group’s underlying EBITDA as a percentage of Group Revenue, adjusted to exclude certain items to better reflect the performance of the Group’s mining business. The Mining EBITDA margin reflects Debswana accounting treatment as a 50/50

  Rank 3 | id=tbl_p1911 | sim=0.5503
  De Beers The annual impairment assessment 

In [73]:
# What is tbl_0361's dense rank?
q_vec = minilm.encode("What was the underlying EBIT for De Beers in 2019?", normalize_embeddings=True)
dense_sims = np.dot(chunk_embeddings["table_aware"], q_vec)
dense_order = np.argsort(dense_sims)[::-1]
chunk_ids = [c["id"] for c in all_chunks["table_aware"]]

for target in ["tbl_0361", "tbl_0377"]:
    idx = chunk_ids.index(target)
    rank = int(np.where(dense_order == idx)[0][0]) + 1
    print(f"{target}: dense rank={rank}/{len(chunk_ids)}, sim={dense_sims[idx]:.4f}")

# Try different RRF_K values
for test_k in [5, 10, 20, 60]:
    rrf = np.zeros(len(chunk_ids))
    for rp, di in enumerate(dense_order):
        rrf[di] += 1.0 / (test_k + rp + 1)
    bm25_order = np.argsort(bm25.score("What was the underlying EBIT for De Beers in 2019?"))[::-1]
    for rp, di in enumerate(bm25_order):
        rrf[di] += 1.0 / (test_k + rp + 1)
    top5 = np.argsort(rrf)[::-1][:5]
    print(f"\nRRF_K={test_k}: top-5 = {[chunk_ids[i] for i in top5]}")

tbl_0361: dense rank=90/2348, sim=0.3321
tbl_0377: dense rank=106/2348, sim=0.3321

RRF_K=5: top-5 = ['tbl_p1757', 'tbl_p2291', 'tbl_p1766', 'tbl_p2109', 'tbl_p1911']

RRF_K=10: top-5 = ['tbl_p1757', 'tbl_p1766', 'tbl_p2109', 'tbl_p2291', 'tbl_p1911']

RRF_K=20: top-5 = ['tbl_p1757', 'tbl_p1766', 'tbl_p2109', 'tbl_p2291', 'tbl_p1911']

RRF_K=60: top-5 = ['tbl_p1757', 'tbl_p1766', 'tbl_p2109', 'tbl_p2291', 'tbl_p1911']


## 10. Decision documentation template

Fill this in after reviewing the results and save to `CONTRIBUTING.md`.

In [65]:
# Auto-populate template from results
if not summary_df.empty:
    best_strat   = summary_df.index[0]
    best_recall  = summary_df.loc[best_strat, f"recall@{EVAL_K}"]
    best_mrr     = summary_df.loc[best_strat, "mrr"]

    template = f"""## ADR: Chunking Strategy Selection

**Date:** [fill in]
**Status:** Accepted

### Context
Six chunking strategies were evaluated against MiniLM-L6-v2 embeddings using
hand-crafted ground truth snippets ({len(raw_gt)} queries, Recall@{EVAL_K} + MRR).
Documents: ANTAM and Anglo American Carbon Performance disclosures.

### Results summary
{summary_df.to_string()}

### Decision
Selected strategy: **{best_strat}**
Mean Recall@{EVAL_K}: {best_recall:.4f} | Mean MRR: {best_mrr:.4f}

### Rationale
[Fill in: why this strategy outperformed — e.g. table-aware routing prevented
mid-row splits; token_aware better matched model context window; etc.]

### Limitations
[Fill in: e.g. paragraph strategy degraded on spatial extraction output;
element_type depends on Unstructured header detection quality; etc.]

### Artefacts
- Evaluation notebook: `chunking_evaluation.ipynb`
- Per-query results: `{per_query_path}`
- Summary: `{summary_path}`
- Ground truth: `{GROUND_TRUTH_PATH}`
"""
    print(template)

## ADR: Chunking Strategy Selection

**Date:** [fill in]
**Status:** Accepted

### Context
Six chunking strategies were evaluated against MiniLM-L6-v2 embeddings using
hand-crafted ground truth snippets (6 queries, Recall@5 + MRR).
Documents: ANTAM and Anglo American Carbon Performance disclosures.

### Results summary
                  recall@5     mrr
strategy                          
element_type           1.0  0.3567
paragraph              0.8  0.4167
token_aware            0.8  0.5400
char_limit             0.6  0.2900
table_aware            0.5  0.2417
sentence_sliding       0.4  0.3333

### Decision
Selected strategy: **element_type**
Mean Recall@5: 1.0000 | Mean MRR: 0.3567

### Rationale
[Fill in: why this strategy outperformed — e.g. table-aware routing prevented
mid-row splits; token_aware better matched model context window; etc.]

### Limitations
[Fill in: e.g. paragraph strategy degraded on spatial extraction output;
element_type depends on Unstructured header detection 